# BookNLP Sharded Batch Processing

This notebook preserves the sharded Colab workflow used to run BookNLP over a Project Gutenberg ID list. It installs BookNLP in an isolated Python 3.10 environment, downloads and conservatively cleans source texts, assigns books to reproducible shards, and copies only completed compressed outputs to Google Drive. Historical Colab and Drive paths are retained as research provenance. This notebook does not implement the later paragraph-reconstruction or 100-bin passage-sampling stages, which are represented by frozen derived data.

**Runtime requirement:** select a T4 GPU in Colab before processing.

### Output layout

```
Drive/booknlp_batch/output/<gutenberg_id>/<gutenberg_id>.tokens.gz
                                          <gutenberg_id>.entities.gz
                                          <gutenberg_id>.quotes.gz
                                          <gutenberg_id>.supersense.gz
                                          <gutenberg_id>.book
                                          _COMPLETE
```

### Sharded execution

Set `N_SHARDS = 2` and assign each worker a distinct `SHARD` value. Books are assigned round-robin, so workers use the same unsplit CSV and can merge completed outputs afterward.

### Conservative text cleaning

The workflow removes Project Gutenberg headers and footers, normalizes line endings and Unicode, removes `_italic_` markers, and collapses repeated blank lines. Front matter, chapter headings, section dividers, and narrative text are retained.


## Cell 1 — Check the runtime

In [2]:
!python --version
!nvidia-smi -L || echo "No GPU attached - it will still run, just slower"

Python 3.12.13
/bin/bash: line 1: nvidia-smi: command not found
No GPU attached - it will still run, just slower


## Cell 2 — Build a Python 3.10 environment and install BookNLP

BookNLP pins `transformers<=4.30.0`, which requires `tokenizers 0.13.3`. Because that tokenizer version does not provide a Python 3.12 wheel, the workflow creates an isolated Python 3.10 environment with `uv`. The `--seed` option installs `setuptools`, which BookNLP requires through `pkg_resources`. This setup usually takes three to five minutes.


In [ ]:
!pip install -q uv

# Create a Python 3.10 venv (uv downloads a standalone CPython 3.10 automatically)
!uv venv /content/bnlp --python 3.10 --seed

PY = "/content/bnlp/bin/python"

# numpy<2   : transformers 4.30 is not compatible with numpy 2.x
# torch 2.4.1 : the linux wheel on PyPI bundles CUDA, so Colab's GPU works out of the box
!uv pip install --python {PY} "numpy<2" "torch==2.4.1" "spacy==3.7.5" \
    "transformers==4.30.0" "tokenizers<0.14" "huggingface_hub<0.20" click booknlp
# spaCy English model (BookNLP uses it for POS tagging, dependency parsing, sentence splitting)
!{PY} -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.0/22.0 MB 46.1 MB/s eta 0:00:00
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment with seed packages at: bnlp
 + packaging==26.2
 + pip==26.1.2
 + setuptools==83.0.0
 + wheel==0.47.0
Activate with: source bnlp/bin/activate
Using Python 3.10.12 environment at: bnlp
Resolved 86 packages in 637ms
Prepared 83 packages in 1m 00s
Installed 83 packages in 869ms
 + absl-py==2.5.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + astunparse==1.6.3
 + blis==0.7.11
 + booknlp==1.0.8
 + catalogue==2.0.10
 + certifi==2026.7.22
 + charset-normalizer==3.4.9
 + click==8.4.2
 + cloudpathlib==0.24.0
 + confection==0.1.5
 + cymem==2.0.13
 + filelock==3.32.0
 + flatbuffers==25.12.19
 + fsspec==2026.7.0
 + gast==0.7.0
 + google-pasta==0.2.0
 + grpcio==1.83.0
 + h5py==3.14.0
 + huggingface-hub==0.19.4
 + idna==3.18
 + jinja2==3.1.6
 + keras==3.12.3
 + langcodes==3.5.1
 + libclang==18.1.1
 + markdown-it-py==4.2.0
 + markup

In [ ]:
# Re-pin: installing the spaCy model can drag transformers/tokenizers back up.
# Install the setuptools pin once for the later run cell.
PY = "/content/bnlp/bin/python"
!uv pip install --python {PY} "transformers==4.30.0" "tokenizers<0.14" "huggingface_hub<0.20" "setuptools<82"
!uv pip uninstall --python {PY} tensorflow

Using Python 3.10.12 environment at: bnlp
Resolved 18 packages in 92ms
Prepared 1 package in 81ms
Uninstalled 1 package in 7ms
Installed 1 package in 8ms
 - setuptools==83.0.0
 + setuptools==81.0.0
Using Python 3.10.12 environment at: bnlp
Uninstalled 1 package in 319ms
 - tensorflow==2.21.0


In [ ]:
%%writefile /content/verify.py
import torch, transformers, spacy, booknlp, pkg_resources
print("torch       ", torch.__version__, "| cuda:", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("spacy       ", spacy.__version__)
nlp = spacy.load("en_core_web_sm")
print("en_core_web_sm OK |", [(t.text, t.pos_) for t in nlp("She wondered if he knew.")][:3])
print("pkg_resources OK")

Writing /content/verify.py


### Verify the install

In [ ]:
PY = "/content/bnlp/bin/python"
!{PY} /content/verify.py

/content/verify.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import torch, transformers, spacy, booknlp, pkg_resources
torch        2.4.1+cu121 | cuda: False
transformers 4.30.0
spacy        3.7.5
en_core_web_sm OK | [('She', 'PRON'), ('wondered', 'VERB'), ('if', 'SCONJ')]
pkg_resources OK


## Cell 3 — Configuration

### Persistent storage

`ROOT` is stored on Google Drive so completed work survives Colab disconnections. Each book is processed in local scratch space and copied to Drive only after `process()` succeeds; the `_COMPLETE` marker distinguishes finished books from partial attempts. Cleaned source texts are also persisted on Drive.

### Sharding

`SHARD` and `N_SHARDS` divide the shared book list round-robin. `SHARD = 1, N_SHARDS = 1` processes the entire list on one worker.

### Output size

The workflow compresses TSV outputs before copying them to Drive and omits the `.book.html` browser rendering. These settings reduce storage use without changing the tabular BookNLP outputs consumed downstream.


In [ ]:
# --- shard assignment ---------------------------------------------------------
SHARD    = 1        # shard 1 for one worker; shard 2 for the second worker
N_SHARDS = 2        # set to 1 to run the whole list on one machine

# --- storage ------------------------------------------------------------------
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/Interiority_26Spring/Data/booknlp_batch"
else:
    ROOT = "/content/booknlp_batch"

import os
CLEAN_DIR  = os.path.join(ROOT, "clean")          # cleaned .txt, one per book
OUTPUT_DIR = os.path.join(ROOT, "output")         # final output, one folder per book
WORK_DIR   = "/content/work"                      # local scratch, never on Drive
MANIFEST   = os.path.join(ROOT, f"manifest_shard{SHARD}of{N_SHARDS}.csv")
BATCH_LOG  = os.path.join(ROOT, f"batch_log_shard{SHARD}of{N_SHARDS}.csv")
for d in (CLEAN_DIR, OUTPUT_DIR, WORK_DIR):
    os.makedirs(d, exist_ok=True)

# --- BookNLP settings (same as the single-book notebook) ----------------------
# supersense gives you verb.cognition / verb.emotion / noun.cognition, so keep it
# even though it is the slow part.
PIPELINE = "entity,quote,supersense,event,coref"
MODEL    = "big"          # "small" is ~2x faster and noticeably worse at coref

# --- output size --------------------------------------------------------------
GZIP_TSV    = True        # .tokens -> .tokens.gz ; pandas reads it transparently
DROP_HTML   = True        # discard the .book.html render

# --- download politeness ------------------------------------------------------
DOWNLOAD_DELAY = 1.5
MAX_RETRIES    = 3

print(f"shard {SHARD} of {N_SHARDS}")
print("output ->", OUTPUT_DIR)
print("scratch->", WORK_DIR)

## Cell 4 — Load the input CSV and select this shard

Upload the shared unsplit CSV or set `CSV_PATH` to its Drive location. The shard filter divides the work. The Gutenberg ID column is detected from common names such as `gutenberg_id`, `gid`, `pg_id`, or `id`; title and author columns are optional.


Optional: write one CSV per shard to Drive when collaborators need pre-split input files. Each recipient then runs that file with `SHARD = 1, N_SHARDS = 1`.


In [ ]:
# for s in range(1, N_SHARDS + 1):
#     sub = pd.DataFrame([b for i, b in enumerate(all_books) if i % N_SHARDS == s - 1])
#     path = os.path.join(ROOT, f"books_shard{s}of{N_SHARDS}.csv")
#     sub.to_csv(path, index=False)
#     print(len(sub), "books ->", path)

## Cell 5 — Download and clean

Deliberately conservative. Only two things get removed, and neither is narrative
text:

1. **The Project Gutenberg header and footer**, matched on the standard
   `*** START OF THE PROJECT GUTENBERG EBOOK ... ***` markers. If a marker is
   missing the text is kept whole and flagged in the manifest rather than cut on
   a guess.
2. **`_italic_` underscore markers**, because spaCy tokenizes `_that_` oddly.
   The word survives, only the underscores go.

Then line endings are normalised, runs of 3+ blank lines collapse to one blank
line, and Unicode is NFC-normalised.

Front matter, title pages, tables of contents, chapter headings, and `* * *`
dividers all stay — your pipeline handles those later. The one consequence worth
knowing: they consume `paragraph_ID`s, so paragraph 0 is often not the first
paragraph of the novel.

Downloads try `aleph.gutenberg.org` and `gutenberg.pglaf.org` before
`www.gutenberg.org` — PG asks bulk downloaders to use mirrors, and Colab's
shared IP is exactly the kind that gets blocked. Already-cleaned books are
skipped, so re-running costs nothing.

## Cell 6 — Quick sanity check

Not looking for clean openings — front matter is expected. Looking for the two
things that would waste GPU hours: a book whose length is wildly off (wrong
edition, a stub, an abridgement) and a book where the PG markers were missing,
which usually means the license text is still attached.

## Cell 7 — Batch processing

`BookNLP()` is constructed once outside the loop to avoid reloading model weights for every book. Each book is processed in local scratch space, its TSV files are compressed, its completed directory is copied to Drive, and `_COMPLETE` is written before scratch files are removed. Interrupted books lack the completion marker and are processed again on the next run.

The initial run downloads BookNLP model weights to `/root/booknlp_models/`. `BNLP_HOURS` can stop the loop cleanly before a Colab session limit.


In [ ]:
%%writefile /content/run_booknlp_batch.py
import csv, gc, gzip, os, shutil, sys, time, traceback

MANIFEST, OUTPUT_DIR, WORK_DIR, LOG_PATH = sys.argv[1:5]

PIPELINE  = os.environ.get("BNLP_PIPELINE", "entity,quote,supersense,event,coref")
MODEL     = os.environ.get("BNLP_MODEL", "big")
BUDGET    = float(os.environ.get("BNLP_HOURS", "0")) * 3600     # 0 = no limit
GZIP_TSV  = os.environ.get("BNLP_GZIP", "1") == "1"
DROP_HTML = os.environ.get("BNLP_DROP_HTML", "1") == "1"

import torch
from booknlp.booknlp import BookNLP

TSV = (".tokens", ".entities", ".quotes", ".supersense")

def publish(workdir, destdir):
    """Copy a finished book to Drive, compressing on the way. Marker written last."""
    tmp = destdir + ".partial"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp, exist_ok=True)
    for fn in sorted(os.listdir(workdir)):
        src = os.path.join(workdir, fn)
        if DROP_HTML and fn.endswith(".html"):
            continue
        if GZIP_TSV and fn.endswith(TSV):
            with open(src, "rb") as fi, gzip.open(os.path.join(tmp, fn + ".gz"), "wb", 6) as fo:
                shutil.copyfileobj(fi, fo, 1024 * 1024)
        else:
            shutil.copy2(src, os.path.join(tmp, fn))
    shutil.rmtree(destdir, ignore_errors=True)
    os.rename(tmp, destdir)
    open(os.path.join(destdir, "_COMPLETE"), "w").close()

with open(MANIFEST, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

todo = [r for r in rows
        if not os.path.exists(os.path.join(OUTPUT_DIR, r["gutenberg_id"], "_COMPLETE"))]

print(f"{len(rows)} in manifest | {len(rows) - len(todo)} already complete | "
      f"{len(todo)} to run", flush=True)
print(f"model={MODEL}  pipeline={PIPELINE}  cuda={torch.cuda.is_available()}  "
      f"gzip={GZIP_TSV}", flush=True)
if not todo:
    print("nothing to do"); sys.exit(0)

t0 = time.time()
booknlp = BookNLP("en", {"pipeline": PIPELINE, "model": MODEL})
print(f"models loaded in {time.time() - t0:.0f}s", flush=True)

new_log = not os.path.exists(LOG_PATH)
log  = open(LOG_PATH, "a", newline="", encoding="utf-8")
logw = csv.writer(log)
if new_log:
    logw.writerow(["gutenberg_id", "status", "seconds", "words", "error", "finished_at"])
    log.flush()

start, ok, err = time.time(), 0, 0

for i, r in enumerate(todo, 1):
    gid   = r["gutenberg_id"]
    words = r.get("words", "")
    work  = os.path.join(WORK_DIR, gid)
    dest  = os.path.join(OUTPUT_DIR, gid)

    if BUDGET and (time.time() - start) > BUDGET:
        print(f"\ntime budget reached - stopping cleanly, {len(todo) - i + 1} books "
              f"left. Re-run this cell to continue.", flush=True)
        break

    print(f"[{i}/{len(todo)}] {gid} ({words} words) ...", end=" ", flush=True)
    t = time.time()
    try:
        shutil.rmtree(work, ignore_errors=True)
        os.makedirs(work, exist_ok=True)
        booknlp.process(r["clean_path"], work, gid)
        publish(work, dest)
        dt = time.time() - t; ok += 1
        print(f"done in {dt/60:.1f} min", flush=True)
        logw.writerow([gid, "ok", round(dt, 1), words, "", time.strftime("%F %T")])
    except Exception as e:
        dt = time.time() - t; err += 1
        print(f"FAILED after {dt/60:.1f} min: {type(e).__name__}: {e}", flush=True)
        traceback.print_exc()
        logw.writerow([gid, "failed", round(dt, 1), words,
                       f"{type(e).__name__}: {e}", time.strftime("%F %T")])
    finally:
        shutil.rmtree(work, ignore_errors=True)
        log.flush()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    el = time.time() - start
    print(f"    elapsed {el/3600:.2f}h | avg {el/i/60:.1f} min/book | "
          f"eta {(len(todo)-i)*el/i/3600:.1f}h", flush=True)

log.close()
print(f"\nBATCH DONE  ok={ok}  failed={err}", flush=True)

Run the batch cell. It reports progress and a running estimate and resumes from completed Drive outputs after a disconnection.


In [ ]:
PY = "/content/bnlp/bin/python"
import os
os.environ["BNLP_PIPELINE"]  = PIPELINE
os.environ["BNLP_MODEL"]     = MODEL
os.environ["BNLP_HOURS"]     = "11"                     # "0" for no limit
os.environ["BNLP_GZIP"]      = "1" if GZIP_TSV  else "0"
os.environ["BNLP_DROP_HTML"] = "1" if DROP_HTML else "0"

!{PY} /content/run_booknlp_batch.py "{MANIFEST}" "{OUTPUT_DIR}" "{WORK_DIR}" "{BATCH_LOG}"

## Cell 8 — Progress and failures

In [ ]:
import pandas as pd, os, glob

done = {os.path.basename(os.path.dirname(p))
        for p in glob.glob(os.path.join(OUTPUT_DIR, "*", "_COMPLETE"))}
man  = pd.read_csv(MANIFEST, dtype={"gutenberg_id": str})
mine = [g for g in man.gutenberg_id if g in done]
print(f"this shard: {len(mine)} / {len(man)} complete")
print(f"in output folder overall (incl. merged): {len(done)}")

missing = [g for g in man.gutenberg_id if g not in done]
print(f"\n{len(missing)} remaining:", missing[:20], "..." if len(missing) > 20 else "")

if os.path.exists(BATCH_LOG):
    log = pd.read_csv(BATCH_LOG, dtype={"gutenberg_id": str})
    print("\nstatus:"); print(log.status.value_counts().to_string())
    okr = log[log.status == "ok"]
    if len(okr):
        print(f"median {okr.seconds.median()/60:.1f} min/book | "
              f"projected for {len(missing)} left: {okr.seconds.median()*len(missing)/3600:.1f}h")
    bad = log[log.status == "failed"]
    if len(bad):
        print("\nfailures:")
        print(bad[["gutenberg_id", "error"]].to_string(index=False))

# retry a failure by deleting its folder, then re-run Cell 7:
# import shutil; shutil.rmtree(os.path.join(OUTPUT_DIR, "12345"), ignore_errors=True)

## Cell 9 — Merge a collaborator's shard

Point `OTHER` to the collaborator's shared output directory. Only directories with `_COMPLETE` are copied, and existing completed directories are not overwritten.


In [ ]:
import shutil, glob, os

OTHER = "/content/drive/MyDrive/booknlp_batch_friend/output"   # Collaborator output folder

copied = skipped = 0
for marker in glob.glob(os.path.join(OTHER, "*", "_COMPLETE")):
    src = os.path.dirname(marker)
    gid = os.path.basename(src)
    dst = os.path.join(OUTPUT_DIR, gid)
    if os.path.exists(os.path.join(dst, "_COMPLETE")):
        skipped += 1; continue
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst)
    copied += 1
    print("merged", gid, flush=True)

print(f"\ncopied {copied}, already had {skipped}")
print("total complete:", len(glob.glob(os.path.join(OUTPUT_DIR, "*", "_COMPLETE"))))

## Cell 10 — Load the output

Runs in Colab's default Python 3.12 with pandas; the venv is not needed since
BookNLP writes plain TSV and JSON. `read_csv` decompresses `.gz` on its own, so
this is the same code you would write against uncompressed output.

Two things about `.tokens` that the README gets wrong:

- The README lists 12 columns; the code actually writes **13** (there is an extra `fine_POS_tag`).
- `byte_onset` / `byte_offset` are misnamed. The source stores spaCy's `tok.idx`, which is a
  **character** offset. So slice the original text directly with `text[onset:offset]` —
  do not `.encode()` first.

`quoting=3` (QUOTE_NONE) is essential — otherwise the novel's quotation marks swallow whole rows.

In [ ]:
import pandas as pd, os, glob

def load_book(gid, exts=(".tokens", ".entities", ".quotes", ".supersense")):
    """Load one book's BookNLP output. Handles gzipped and plain files."""
    gid = str(gid)
    out = {}
    for e in exts:
        p = os.path.join(OUTPUT_DIR, gid, gid + e)
        p = p if os.path.exists(p) else p + ".gz"
        out[e.lstrip(".")] = pd.read_csv(p, sep="\t", quoting=3, keep_default_na=False)
    return out

gid = sorted(done)[0]
d = load_book(gid)
for k, v in d.items():
    print(f"{k:<11}", v.shape)
print("columns:", list(d["tokens"].columns))
print("paragraphs:", d["tokens"].paragraph_ID.nunique())
d["tokens"].head(10)

The loaded outputs can support corpus-wide feature extraction, such as cognition- and emotion-supersense rates. This example is illustrative and is not the later passage-reconstruction or 100-bin sampling workflow.

```python
rows = []
for gid in sorted(done):
    b  = load_book(gid, exts=(".tokens", ".supersense"))
    n  = len(b["tokens"])
    c  = b["supersense"].supersense_category.value_counts()
    rows.append({"gutenberg_id": gid, "tokens": n,
                 **{k: c.get(k, 0) / n * 1000
                    for k in ("verb.cognition", "verb.emotion", "noun.cognition")}})
corpus = pd.DataFrame(rows)
```
